### Section 0 - Setup

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
from fredapi import Fred
from dotenv import load_dotenv
import os

In [18]:
#Loads API key from .env file
load_dotenv()
api_key = os.getenv("FRED_API_KEY")
fred = Fred(api_key=api_key)

In [19]:
test_1 = fred.get_series("DGS10", observation_start="2024-01-01", observation_end="2024-01-10")
print(test_1)

2024-01-01     NaN
2024-01-02    3.95
2024-01-03    3.91
2024-01-04    3.99
2024-01-05    4.05
2024-01-08    4.01
2024-01-09    4.02
2024-01-10    4.04
dtype: float64


After importing the relevant packages, I can load the API key from the .env file, and use the fredapi package to connect to the API. After running the second block of code without any errors, I used the get_series function to collect a sample of data between Jan 1-10 2024,from DGS10 for further verification. I can see from the output that it looks consistent with expected yield levels(~4%).

### Section 1 - Data Collection and Validation

In [20]:
try:
    dgs10 = fred.get_series("DGS10", observation_start="1800-01-01")
    dgs10 = dgs10.rename("DGS10")
except Exception as e:
    print(f"Failed to pull DGS10: {e}")

print("Shape:", dgs10.shape)
print(dgs10.head())
print(dgs10.tail())
print("Index type:", type(dgs10.index))
print("Missing values:", dgs10.isna().sum())
print("First 10 missing dates:", dgs10[dgs10.isna()].index[:10].tolist())
print("Date spacing (top 5):")
print(dgs10.index.to_series().diff().value_counts().head())
print("Min/Max:", dgs10.min(), "/", dgs10.max())

dgs10.to_csv("raw_dgs10.csv")

Shape: (16861,)
1962-01-02    4.06
1962-01-03    4.03
1962-01-04    3.99
1962-01-05    4.02
1962-01-08    4.03
Name: DGS10, dtype: float64
2026-08-12    4.68
2026-08-13    4.63
2026-08-14    4.68
2026-08-17    4.72
2026-08-18    4.71
Name: DGS10, dtype: float64
Index type: <class 'pandas.DatetimeIndex'>
Missing values: 719
First 10 missing dates: [Timestamp('1962-02-12 00:00:00'), Timestamp('1962-02-22 00:00:00'), Timestamp('1962-04-20 00:00:00'), Timestamp('1962-05-30 00:00:00'), Timestamp('1962-07-04 00:00:00'), Timestamp('1962-09-03 00:00:00'), Timestamp('1962-10-12 00:00:00'), Timestamp('1962-11-06 00:00:00'), Timestamp('1962-11-12 00:00:00'), Timestamp('1962-11-22 00:00:00')]
Date spacing (top 5):
1 days    13488
3 days     3372
Name: count, dtype: int64
Min/Max: 0.52 / 15.84


What we can see from the output:

- I set `observation_start` to 1800-01-01 (deliberately before any real data) so FRED reveals the true start date rather than one I assume - this confirms DGS10 begins 1962-01-02. I rename it to "DGS10" for clarity in later steps.

- Shape (16,860 rows) matches ~64 years of daily, business day data. Index and values are datetime and float, as expected. The 719 missing values are all US federal holidays (confirmed by the first 10 dates), not a data issue.

- Date spacing only shows 1-day and 3-day gaps because `.diff()` checks the index, not values: weekends are excluded from the index (a real 3-day gap), while holidays still exist as rows with a NaN value (no gap at all).

- The min/max values are plausible values so we can confirm that there are no sentinel values. 

In [21]:
try:
    dgs2 = fred.get_series("DGS2", observation_start="1800-01-01")
    dgs2 = dgs2.rename("DGS2")
except Exception as e:
    print(f"Failed to pull DGS2: {e}")

print("Shape:", dgs2.shape)
print(dgs2.head())
print(dgs2.tail())
print("Index type:", type(dgs2.index))
print("Missing values:", dgs2.isna().sum())
print("First 10 missing dates:", dgs2[dgs2.isna()].index[:10].tolist())
print("Date spacing (top 5):")
print(dgs2.index.to_series().diff().value_counts().head())
print("Min/Max:", dgs2.min(), "/", dgs2.max())

dgs2.to_csv("raw_dgs2.csv")

Shape: (13101,)
1976-06-01    7.26
1976-06-02    7.23
1976-06-03    7.22
1976-06-04    7.12
1976-06-07    7.09
Name: DGS2, dtype: float64
2026-08-12    4.20
2026-08-13    4.15
2026-08-14    4.17
2026-08-17    4.19
2026-08-18    4.19
Name: DGS2, dtype: float64
Index type: <class 'pandas.DatetimeIndex'>
Missing values: 551
First 10 missing dates: [Timestamp('1976-07-05 00:00:00'), Timestamp('1976-09-06 00:00:00'), Timestamp('1976-10-11 00:00:00'), Timestamp('1976-11-02 00:00:00'), Timestamp('1976-11-11 00:00:00'), Timestamp('1976-11-25 00:00:00'), Timestamp('1976-12-24 00:00:00'), Timestamp('1977-02-21 00:00:00'), Timestamp('1977-04-08 00:00:00'), Timestamp('1977-05-30 00:00:00')]
Date spacing (top 5):
1 days    10480
3 days     2620
Name: count, dtype: int64
Min/Max: 0.09 / 16.95


The same methods are used for DGS2, so we can see that the data starts mid-1976, 14 years after DGS10. This limits the 10Y-2Y analysis. After some research I can see that this is because 2-year treasury notes weren't issued on a regular schedule until the mid-1970s. All the other values check out, so we can move on from here. 

In [22]:
try:
    dgs3mo = fred.get_series("DGS3MO", observation_start="1800-01-01")
    dgs3mo = dgs3mo.rename("DGS3MO")
except Exception as e:
    print(f"Failed to pull DGS3MO: {e}")

print("Shape:", dgs3mo.shape)
print(dgs3mo.head())
print(dgs3mo.tail())
print("Index type:", type(dgs3mo.index))
print("Missing values:", dgs3mo.isna().sum())
print("First 10 missing dates:", dgs3mo[dgs3mo.isna()].index[:10].tolist())
print("Date spacing (top 5):")
print(dgs3mo.index.to_series().diff().value_counts().head())
print("Min/Max:", dgs3mo.min(), "/", dgs3mo.max())

dgs3mo.to_csv("raw_dgs3mo.csv")

Shape: (11731,)
1981-09-01    17.01
1981-09-02    16.65
1981-09-03    16.96
1981-09-04    16.64
1981-09-07      NaN
Name: DGS3MO, dtype: float64
2026-08-12    3.87
2026-08-13    3.87
2026-08-14    3.86
2026-08-17    3.87
2026-08-18    3.86
Name: DGS3MO, dtype: float64
Index type: <class 'pandas.DatetimeIndex'>
Missing values: 491
First 10 missing dates: [Timestamp('1981-09-07 00:00:00'), Timestamp('1981-10-12 00:00:00'), Timestamp('1981-11-03 00:00:00'), Timestamp('1981-11-11 00:00:00'), Timestamp('1981-11-26 00:00:00'), Timestamp('1981-12-09 00:00:00'), Timestamp('1981-12-25 00:00:00'), Timestamp('1982-01-01 00:00:00'), Timestamp('1982-02-12 00:00:00'), Timestamp('1982-02-15 00:00:00')]
Date spacing (top 5):
1 days    9384
3 days    2346
Name: count, dtype: int64
Min/Max: 0.0 / 17.01


This is the same check as we did before for DGS10 and DGS2, all the values check out perfectly, and we can see that the data started in 1981, a lot later than the other two, so this will limit our comparision capabilities, as the date ranges differ too much. However, I would still like to use the 10Y-3MO comparision as it's the version the NY Fed's own recession model actually uses and has been shown to outperform 10Y-2Y empirically. 

In [23]:
try:
    usrec = fred.get_series("USREC", observation_start="1800-01-01")
    usrec = usrec.rename("USREC")
except Exception as e:
    print(f"Failed to pull USREC: {e}")

print("Shape:", usrec.shape)
print(usrec.head())
print(usrec.tail())
print("Index type:", type(usrec.index))
print("Missing values:", usrec.isna().sum())
print("Unique values:", usrec.unique())
print("Date spacing (top 5):")
print(usrec.index.to_series().diff().value_counts().head())

usrec.to_csv("raw_usrec.csv")

Shape: (2060,)
1854-12-01    1.0
1855-01-01    0.0
1855-02-01    0.0
1855-03-01    0.0
1855-04-01    0.0
Name: USREC, dtype: float64
2026-03-01    0.0
2026-04-01    0.0
2026-05-01    0.0
2026-06-01    0.0
2026-07-01    0.0
Name: USREC, dtype: float64
Index type: <class 'pandas.DatetimeIndex'>
Missing values: 0
Unique values: [1. 0.]
Date spacing (top 5):
31 days    1201
30 days     686
28 days     130
29 days      42
Name: count, dtype: int64


USREC goes back to **1854** - NBER's recession dating extends way further than the other three yield series, so this isn't the limiting factor for our date range. No missing values, since it's a monthly series with no daily weekend or holiday gaps. The `[1. 0.]` unique values confirm it's a binary flag, as expected. Date spacing (28–31 days) confirms monthly frequency.